# Arrays & Hashing — Technical Reference

## Quick Index

| Technique | When to use | Problems |
| :--- | :--- | :--- |
| Prefix Sum | Range sum queries after O(n) preprocessing | 303, 560, 724 |
| 2D Prefix Sum | Range sum queries on a matrix | 304, 1314 |
| Difference Array | Range increment/decrement updates in O(1) | 1109, 1854 |
| Index-as-Hash | Encode presence using array indices — no extra space | 41, 442, 448 |
| Frequency Map | Count occurrences; detect anagrams, majority element | 242, 169, 347 |
| KMP | Pattern matching in O(n + m) — no backtracking on text | 28, 459 |
| Expand from Center | Find longest palindromic substring in O(n²) | 5, 647 |

---
## Prefix Sum

Precompute cumulative sums so any range sum `[i, j]` is answered in O(1). Use when the same array is queried multiple times.

In [ ]:
# Build prefix sum — O(n)
# prefix[i] = sum of nums[0..i-1]  (1-indexed offset avoids boundary checks)
def build_prefix(nums):
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n):
        prefix[i + 1] = prefix[i] + nums[i]
    return prefix

# Query range sum [l, r] inclusive — O(1)
def range_sum(prefix, l, r):
    return prefix[r + 1] - prefix[l]


# LC 560 — Subarray Sum Equals K
# Count subarrays summing to k using prefix sum + hashmap
# prefix[j] - prefix[i] == k  →  prefix[i] == prefix[j] - k
from collections import defaultdict
def subarraySum(nums, k):
    count = defaultdict(int)
    count[0] = 1                        # empty prefix sums to 0
    prefix = 0
    res = 0
    for num in nums:
        prefix += num
        res    += count[prefix - k]     # how many prior prefixes allow sum == k
        count[prefix] += 1
    return res

Problems: LC 303, LC 560, LC 724, LC 525

---
## 2D Prefix Sum

Extend prefix sum to a matrix. Any rectangular region sum is answered in O(1) after O(mn) preprocessing.

In [ ]:
# Build 2D prefix sum — O(mn)
# prefix[i][j] = sum of all cells in rectangle (0,0) to (i-1, j-1)
def build_prefix_2d(matrix):
    m, n = len(matrix), len(matrix[0])
    prefix = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            prefix[i][j] = (matrix[i-1][j-1]
                            + prefix[i-1][j]     # above
                            + prefix[i][j-1]     # left
                            - prefix[i-1][j-1])  # subtract double-counted corner
    return prefix

# Query rectangle sum (r1,c1) to (r2,c2) inclusive — O(1)
def region_sum(prefix, r1, c1, r2, c2):
    return (prefix[r2+1][c2+1]
            - prefix[r1][c2+1]
            - prefix[r2+1][c1]
            + prefix[r1][c1])

Problems: LC 304, LC 1314, LC 1738

---
## Difference Array

Apply range increment/decrement updates in O(1) each, then reconstruct the final array in O(n). Use when many range updates are needed before reading the result.

In [ ]:
# Build difference array
def build_diff(nums):
    n = len(nums)
    diff = [0] * (n + 1)
    diff[0] = nums[0]
    for i in range(1, n):
        diff[i] = nums[i] - nums[i-1]
    return diff

# Range update: add val to nums[l..r] — O(1)
def range_update(diff, l, r, val):
    diff[l]     += val      # start of range: increase
    diff[r + 1] -= val      # end of range + 1: cancel increase

# Reconstruct final array from diff — O(n)
def reconstruct(diff):
    result = [0] * len(diff)
    result[0] = diff[0]
    for i in range(1, len(diff)):
        result[i] = result[i-1] + diff[i]
    return result


# LC 1109 — Corporate Flight Bookings
def corpFlightBookings(bookings, n):
    diff = [0] * (n + 2)
    for first, last, seats in bookings:
        diff[first]    += seats
        diff[last + 1] -= seats
    return list(accumulate(diff))[1:n+1]

from itertools import accumulate

Problems: LC 1109, LC 1854, LC 370

---
## Index-as-Hash

Use array indices to encode presence of values — O(1) space when values are in range `[1, n]`. Mark visited by negating `nums[abs(val) - 1]`. Restore by taking absolute values.

In [ ]:
# LC 442 — Find All Duplicates in Array
# Values in [1, n], each appearing once or twice
# Negate nums[val-1] to mark val as seen; if already negative → duplicate
def findDuplicates(nums):
    res = []
    for val in nums:
        idx = abs(val) - 1              # map value to index
        if nums[idx] < 0:
            res.append(abs(val))        # already negated → seen before
        else:
            nums[idx] = -nums[idx]      # negate to mark as seen
    return res


# LC 41 — First Missing Positive
# Place each value val in position val-1 (cyclic sort), then scan for gap
def firstMissingPositive(nums):
    n = len(nums)
    for i in range(n):                  # cyclic sort: put val at index val-1
        while 1 <= nums[i] <= n and nums[nums[i]-1] != nums[i]:
            nums[nums[i]-1], nums[i] = nums[i], nums[nums[i]-1]
    for i in range(n):                  # first gap = first missing positive
        if nums[i] != i + 1:
            return i + 1
    return n + 1

Problems: LC 41, LC 442, LC 448

---
## Frequency Map

Count occurrences using a `Counter` or `defaultdict`. Use for anagram detection, majority element, top-K frequency, and grouping by frequency.

In [ ]:
from collections import Counter, defaultdict

# Anagram detection — compare two Counters
def isAnagram(s, t):
    return Counter(s) == Counter(t)

# Group anagrams — key = sorted word
def groupAnagrams(strs):
    groups = defaultdict(list)
    for s in strs:
        groups[tuple(sorted(s))].append(s)
    return list(groups.values())

# LC 169 — Majority Element (Boyer-Moore Voting)
# Majority element appears > n/2 times
# No extra space — count cancels out non-majority elements
def majorityElement(nums):
    candidate, count = None, 0
    for num in nums:
        if count == 0:
            candidate = num
        count += 1 if num == candidate else -1
    return candidate

Problems: LC 242, LC 49, LC 169, LC 347

---
## KMP — Knuth-Morris-Pratt Pattern Matching

Find all occurrences of a pattern in a text in O(n + m) — no backtracking on the text pointer. The key is the **failure function** (also called LPS — Longest Proper Prefix which is also Suffix), which tells you how far to shift the pattern on a mismatch.

In [ ]:
# Build failure function (LPS array) — O(m)
# lps[i] = length of longest proper prefix of pattern[0..i] that is also a suffix
def build_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length = 0                          # length of previous longest prefix-suffix
    i = 1
    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]    # fall back — don't increment i
        else:
            lps[i] = 0
            i += 1
    return lps

# KMP search — O(n + m)
# Returns list of starting indices where pattern is found in text
def kmp_search(text, pattern):
    n, m = len(text), len(pattern)
    lps = build_lps(pattern)
    res = []
    i = j = 0                           # i = text pointer, j = pattern pointer
    while i < n:
        if text[i] == pattern[j]:
            i += 1; j += 1
        if j == m:                      # full match found
            res.append(i - j)
            j = lps[j - 1]             # shift pattern using LPS
        elif i < n and text[i] != pattern[j]:
            if j != 0:
                j = lps[j - 1]         # mismatch after some match — use LPS
            else:
                i += 1                  # mismatch at start — advance text
    return res

Problems: LC 28, LC 459, LC 686

---
## Expand from Center — Palindrome

For each character (and gap between characters), expand outward while the palindrome holds. O(n²) time, O(1) space. Covers both odd-length and even-length palindromes.

In [ ]:
# Expand from center helper
# Returns (left, right) indices of longest palindrome centered at (l, r)
def expand(s, l, r):
    while l >= 0 and r < len(s) and s[l] == s[r]:
        l -= 1
        r += 1
    return l + 1, r - 1                 # last valid palindrome bounds


# LC 5 — Longest Palindromic Substring
def longestPalindrome(s):
    res_l = res_r = 0
    for i in range(len(s)):
        # odd-length palindrome: center at i
        l, r = expand(s, i, i)
        if r - l > res_r - res_l:
            res_l, res_r = l, r
        # even-length palindrome: center between i and i+1
        l, r = expand(s, i, i + 1)
        if r - l > res_r - res_l:
            res_l, res_r = l, r
    return s[res_l:res_r + 1]


# LC 647 — Palindromic Substrings (count all)
def countSubstrings(s):
    count = 0
    for i in range(len(s)):
        for l, r in [(i, i), (i, i+1)]:    # odd and even centers
            while l >= 0 and r < len(s) and s[l] == s[r]:
                count += 1
                l -= 1; r += 1
    return count

Problems: LC 5, LC 647, LC 516